In [1]:
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="descriptives")

# [read] Create deflator dfs from ONS excel files

### Inflation data
- Import mm23.xlsx from dirs.input_dir, and load sheet 'data'
- The first few rows are filler, so we will need to find the first row where the index is 'CDID' and make that the header
- This is probably row 2, so we should write an assert statement to check that
- Then, filter all columns which are not inflation_cdid = 'L522'
- Rows are either 'filler', 'yearly', 'quarterly' or 'monthly'
- Filter out 'filler' rows. Filter out 'monthly' rows. 'Filter out 'quarterly' rows.
- Filter out 'yearly' rows which are not in the range 2005 - 2026
- Output the remaining series

In [2]:
import pandas as pd

inflation_cdid = 'L522' # CPIH INDEX 0.0
start_year_inf = 2005
xlsx_file = f"{dirs.input_dir}/mm23.xlsx"
xlsx_sheet = "data"

df_raw = pd.read_excel(xlsx_file, sheet_name=xlsx_sheet, engine="calamine")

# Make the first column the index
df_raw.set_index(df_raw.columns[0], inplace=True)

#Make the first row the header
assert df_raw.index[0] == 'CDID', "Expected header not found"
df_raw.columns = df_raw.iloc[0]
df = df_raw[1:]

# Filter all columns where the column name is not our inflation_cdid variable
df = df[df.columns[df.columns == inflation_cdid]]

# Autodetect which index references are 'filler', 'yearly', 'quarterly' or 'monthly', 
# where filler rows don't contain a numeric value within it
filler_rows = df[~df.index.str.contains(r'\d')].index
yearly_rows = df[df.index.str.contains(r'^\d{4}$')].index
quarterly_rows = df[df.index.str.contains(r'^\d{4}Q\d$')].index
monthly_rows = df[df.index.str.contains(r'^\d{4}M\d{2}$')].index

df_yearly_only = df.loc[yearly_rows]
df_yearly_only.index = df_yearly_only.index.astype(int)

# Filter out 'yearly' rows which are not in the range 2005 -
df_year_range = df_yearly_only[(df_yearly_only.index >= start_year_inf)]

# Find the last year in this range which is not a null / NaN
end_year_inf = df_year_range[df_year_range[inflation_cdid].notnull()].index[-1]
df_year_range_filtered = df_year_range[df_year_range.index <= end_year_inf]
# Find where the value is 100
base_year_inf = df_year_range_filtered[df_year_range_filtered[inflation_cdid] == 100].index[0]
if base_year_inf == None:
    raise ValueError("No base year found where the inflation index is 100.")

print(f"Base year: {base_year_inf}, Start year: {start_year_inf}, End year: {end_year_inf}")

df_inflation = df_year_range_filtered[inflation_cdid].astype(float)
print(df_inflation)

Base year: 2015, Start year: 2005, End year: 2025
Title
2005     79.4
2006     81.4
2007     83.3
2008     86.2
2009     87.9
2010     90.1
2011     93.6
2012     96.0
2013     98.2
2014     99.6
2015    100.0
2016    101.0
2017    103.6
2018    106.0
2019    107.8
2020    108.9
2021    111.6
2022    120.5
2023    128.6
2024    132.9
2025    138.0
Name: L522, dtype: float64


### Deflator index

In [3]:
# Write a full block of code to import GDP deflator from input_dir / quarterlynationalaccountsdatatables.xlsx
# Use a similar coding style (i.e. setting variables first, naming pattern)
# Sheet name is 'yearly_variables'
# This is already loaded as panel data. First column is years but also contains quarters
# Filter out quarters, and then grab the 'YBGB' column, which corresponds to the GDP deflator.
# print resulting dataframe

import pandas as pd

deflator_cdid = 'YBGB' # GDP deflator
start_year_def = 2005

xlsx_file = f"{dirs.input_dir}/quarterlynationalaccountsdatatables.xlsx"
xlsx_sheet = "yearly_variables"
df_raw_def = pd.read_excel(xlsx_file, sheet_name=xlsx_sheet, engine="calamine")

df_raw_def.set_index(df_raw_def.columns[0], inplace=True)
df_raw_def.index = df_raw_def.index.astype(str)
regex_pattern = r'^\d{4}(?:\.0)?$'
yearly_rows_def = df_raw_def[df_raw_def.index.str.contains(regex_pattern, na=False)].index

# (Optional) If it caught the ".0", clean it up so the index is purely the year:
df_yearly_only_def = df_raw_def.loc[yearly_rows_def]
df_yearly_only_def.index = df_yearly_only_def.index.str.replace('.0', '', regex=False).astype(int)

# Parse end year
df_year_range_def = df_yearly_only_def[(df_yearly_only_def.index >= start_year_def)]
end_year_def = df_year_range_def[df_year_range_def[deflator_cdid].notnull()].index[-1]
df_year_range_filtered_def = df_year_range_def[df_year_range_def.index <= end_year_def]

# Parse base year
base_year_def = df_year_range_filtered_def[df_year_range_filtered_def[deflator_cdid] == 100].index[0]
if base_year_def == None:
    raise ValueError("No base year found where the GDP deflator is 100.")

print(f"Base year: {base_year_def}, Start year: {start_year_def}, End year: {end_year_def}")

df_deflator = df_year_range_filtered_def[deflator_cdid].astype(float)

print(df_deflator)

Base year: 2023, Start year: 2005, End year: 2025
Unnamed: 0
2005     64.0
2006     65.9
2007     67.3
2008     69.5
2009     70.9
2010     71.8
2011     73.6
2012     74.7
2013     76.3
2014     77.6
2015     78.1
2016     79.4
2017     80.7
2018     82.2
2019     84.3
2020     88.4
2021     89.0
2022     94.0
2023    100.0
2024    103.9
2025    107.7
Name: YBGB, dtype: float64


In [4]:
# Combine the two dataframes for columns to be 'cpih' and 'gdpdef'. Index is years
# set a base_year variable to be 2024
# normalise both columns to 2024

base_year = 2024

df_prices = pd.DataFrame({
    'cpih': df_inflation,
    'gdpdef': df_deflator
})
rebase_dict = {
    'cpih': df_prices.loc[base_year, 'cpih'] / df_prices.loc[base_year_inf, 'cpih'],
    'gdpdef': df_prices.loc[base_year, 'gdpdef'] / df_prices.loc[base_year_def, 'gdpdef']
}
df_prices_rebased = df_prices / pd.Series(rebase_dict)

for col in df_prices_rebased.columns:
    if round(df_prices_rebased.loc[base_year, col], 2) == 100:
        continue
    raise ValueError(f"Expected 100 in {base_year} for {col}, got {df_prices_rebased.loc[base_year, col]}")

display(df_prices_rebased)

,cpih,gdpdef
2005,59.744169,61.597690
2006,61.249059,63.426372
2007,62.678706,64.773821
2008,64.860798,66.891242
2009,66.139955,68.238691
2010,67.795335,69.104909
2011,70.428894,70.837344
2012,72.234763,71.896054
2013,73.890143,73.435996
2014,74.943567,74.687199


# [read/write] Deflate yearly data

- Load raw_schema from the Excel file in the same way we did before
- raw_schema has a 'deflate' column which is a string, corresponding to every (panel) property in fame_yearly
- For every column in fame_yearly, find the appropriate deflator key from that column
- Note that some variables may have no deflator key (column value is None/Null)
- Match it to the deflator in df_prices_rebased
- And then create a new table called 'fame_yearly_deflated'
- Which is the exact same as fame_yearly, but with the values deflated by the appropriate deflator
- If there was no deflator key, then the value should be copied over exactly

In [5]:
# Load raw_schema from the Excel file in the same way we did before
# raw_schema has a 'deflate' column which is a string, corresponding to every (panel) property in fame_yearly
import pandas as pd
import ibis
from utils.f_0_dirs import get_data_dirs

build_dirs = get_data_dirs(segment="build")
schema_path = build_dirs.input_dir / "raw_properties.xlsx"
schema_source = pd.read_excel(schema_path, sheet_name="raw_properties", engine="calamine",
    dtype={
        "from_raw": str,
        "key": str,
        "type": str,
        "fuzzy_mapping": str,
        "keep": str,
        "in_ln_set": "boolean",
        "may_mix": "boolean",
        "deflate": str,
        "description": str
    }
)
# Iterate over columns where type is boolean and fillna with False
for col in schema_source.select_dtypes(include='boolean').columns:
    schema_source[col] = schema_source[col].fillna(False)
schema_raw: pd.DataFrame = schema_source[schema_source["from_raw"].notna()]
schema_yearly: pd.DataFrame = schema_raw[schema_raw["keep"].isin(["yearly", "all"])]
schema_deflate_mapping = schema_yearly[["key", "deflate"]].set_index("key").to_dict()["deflate"]

print("Deflation mapping:")
for key, deflator in schema_deflate_mapping.items():
    print(f"  {key}: {deflator}")

Deflation mapping:
  registered_number: nan
  consolidated: nan
  turnover: gdpdef
  shareholders_funds: gdpdef
  profit_loss_pretax: gdpdef
  employees: nan
  tangibles: gdpdef
  tangibles_land_and_buildings: gdpdef
  tangibles_land_freehold: gdpdef
  tangibles_land_leasehold: gdpdef
  tangibles_fixt_fit: gdpdef
  tangibles_plant_and_vehicles: gdpdef
  tangibles_plant: gdpdef
  tangibles_vehicles: gdpdef
  fixed_other: gdpdef
  intangibles: gdpdef
  investments_other: gdpdef
  fixed_total: gdpdef
  liabilities: gdpdef
  total_assets: gdpdef
  liabilites_lt: gdpdef
  cos: gdpdef
  admin_expenses: gdpdef
  interest_paid: gdpdef
  profit_loss_pretax2: gdpdef
  tax: gdpdef
  dividends: gdpdef
  depreciation: gdpdef
  r_and_d: gdpdef
  remuneration_employees: cpih
  wages: cpih
  social_security_costs: cpih
  pensions_costs: cpih
  other_staff_costs: cpih
  renumeration_directors: gdpdef
  ebitda: gdpdef


- For every column in fame_yearly, find the appropriate deflator key from that column
- Note that some variables may have no deflator key (column value is None/Null)
- Match it to the deflator in df_prices_rebased
- And then create a new table called 'fame_yearly_deflated'
- Which is the exact same as fame_yearly, but with the values deflated by the appropriate deflator
- If there was no deflator key, then the value should be copied over exactly
- Setup complicated sql commands to now get the right deflator column for every column in fame_yearly, and then deflate it by the appropriate deflator

In [8]:
table_name = "working_yearly_kp"

start_time = pd.Timestamp.now()
con = ibis.duckdb.connect(str(build_dirs.db_path))

# --- 0. Memory Safeguards (Crucial for 8GB RAM limits) ---
# Tell DuckDB it is not allowed to use more than 4GB of RAM for operations. 
# It will automatically spill intermediate calculations to a temporary disk file.
con.raw_sql("PRAGMA memory_limit='4GB'")

table_fame_yearly = con.table("fame_yearly")

# --- 1. Push Prices Data into DuckDB ---
if df_prices_rebased.index.name != 'year':
    df_prices_rebased.index.name = 'year'
df_prices_rebased.index = df_prices_rebased.index.astype(int)
df_prices_reset = df_prices_rebased.reset_index()

print("📥 Loading prices data into DuckDB...")
prices_table = con.create_table(table_name + "_temp", df_prices_reset, temp=True, overwrite=True)

# --- 2. Build Dynamic Column Selections ---
joined_table = table_fame_yearly.left_join(prices_table, "year")
select_exprs = []

print("⚙️  Building dynamic inflation adjustment abstract syntax tree...")
for col_name in table_fame_yearly.columns:
    deflator = schema_deflate_mapping.get(col_name)
    has_deflator = pd.notna(deflator) and isinstance(deflator, str) and deflator.strip().lower() not in ['nan', 'none', '']
    
    if has_deflator and deflator in prices_table.columns:
        deflated_col = (joined_table[col_name] / (joined_table[deflator] / 100)).name(col_name)
        select_exprs.append(deflated_col)
    else:
        select_exprs.append(joined_table[col_name])

# Build the blueprint of the table (lazy evaluated, no math executed yet)
deflated_table_expr = joined_table.select(*select_exprs)

# --- 3. Create an Empty Target Table ---
print(f"🏗️  Creating empty target table '{table_name}'...")
# Grab 0 rows to define the final schema and instantiate it
empty_schema = deflated_table_expr.filter(deflated_table_expr.year == -9999)
con.create_table(table_name, empty_schema, overwrite=True)

# --- 4. Process in Memory-Safe Batches (By Year) ---
unique_years = df_prices_reset['year'].unique().tolist()
print(f"🔄 Starting batched deflation across {len(unique_years)} years...")

for y in sorted(unique_years):
    print(f"   [{(pd.Timestamp.now() - start_time).total_seconds():.1f}s] Processing & Materializing Year: {y}")
    
    # Filter the lazy expression to just THIS year
    year_chunk = deflated_table_expr.filter(deflated_table_expr.year == y)
    
    # Execute the chunk and APPEND it to the new table
    con.insert(table_name, year_chunk)

# --- 5. Verification ---
print(f"   [{(pd.Timestamp.now() - start_time).total_seconds():.1f}s] Running final checks...")
old_count = table_fame_yearly.count().execute()
new_count = con.table(table_name).count().execute()
print(f"✅ Deflation complete! Materialized {new_count:,} rows (matches original: {old_count == new_count}).")

# --- 6. Generate Verification Samples ---
print("🎲 Generating verification sample...")
# Ibis .sample() can be unpredictable across backends, order_by(random) is bulletproof
sample_yearly_df = con.table("fame_yearly").order_by(ibis.random()).limit(20).execute()

# Rather than a complex .isin() which triggers full table scans on 14 million rows,
# we turn the 20 sample rows into a temporary in-memory table and inner_join it to pluck them out.
keys_to_fetch = ibis.memtable(sample_yearly_df[["registered_number", "year"]])
sample_deflated_df = (
    con.table(table_name)
    .inner_join(keys_to_fetch, ["registered_number", "year"])
    .execute()
)
# Sort this by the same order as the original sample for easy visual comparison
sample_deflated_df = sample_deflated_df.set_index(["registered_number", "year"]).loc[sample_yearly_df.set_index(["registered_number", "year"]).index].reset_index()

display(sample_yearly_df)
display(sample_deflated_df)

📥 Loading prices data into DuckDB...
⚙️  Building dynamic inflation adjustment abstract syntax tree...
🏗️  Creating empty target table 'working_yearly_kp'...
🔄 Starting batched deflation across 21 years...
   [0.0s] Processing & Materializing Year: 2005
   [0.1s] Processing & Materializing Year: 2006
   [3.3s] Processing & Materializing Year: 2007
   [6.2s] Processing & Materializing Year: 2008
   [9.0s] Processing & Materializing Year: 2009
   [11.8s] Processing & Materializing Year: 2010
   [15.0s] Processing & Materializing Year: 2011
   [18.1s] Processing & Materializing Year: 2012
   [21.0s] Processing & Materializing Year: 2013
   [24.7s] Processing & Materializing Year: 2014
   [28.3s] Processing & Materializing Year: 2015
   [32.2s] Processing & Materializing Year: 2016
   [35.7s] Processing & Materializing Year: 2017
   [40.0s] Processing & Materializing Year: 2018
   [44.5s] Processing & Materializing Year: 2019
   [48.8s] Processing & Materializing Year: 2020
   [52.8s] Proc

,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,dividends,depreciation,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda
0,08239651,2015,False,NaN,23.963000,NaN,NaN,96.716,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,09892814,2016,False,NaN,0.001000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,05601468,2006,False,NaN,0.572000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01194193,2017,False,43257.190000,14506.065000,750.585000,180.0,2847.574,2150.709,2150.709,...,NaN,330.802,NaN,8104.043,7178.131,822.299,103.613,NaN,NaN,1556.943000
4,07090289,2011,False,NaN,0.001000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,07134885,2016,False,29.363000,-33.736000,-30.146000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-30.146000
6,IE170959,2023,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,06118398,2012,False,NaN,-16.630000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,13577904,2022,False,NaN,0.500000,NaN,7.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,05020019,2009,False,19145.358000,15877.288000,5538.785000,109.0,NaN,NaN,NaN,...,NaN,NaN,NaN,245.570,215.610,21.108,8.852,NaN,NaN,13749.432000


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,dividends,depreciation,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda
0,08239651,2015,False,NaN,31.879074,NaN,NaN,128.665716,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,09892814,2016,False,NaN,0.001309,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,05601468,2006,False,NaN,0.901833,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01194193,2017,False,55692.962094,18676.333996,966.366561,180.0,3666.207418,2769.004524,2769.004524,...,NaN,425.902451,NaN,10396.016551,9208.239478,1054.860397,132.916677,NaN,NaN,2004.539996
4,07090289,2011,False,NaN,0.001412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,07134885,2016,False,38.423372,-44.145723,-39.447977,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-39.447977
6,IE170959,2023,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,06118398,2012,False,NaN,-23.130616,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,13577904,2022,False,NaN,0.552660,NaN,7.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,05020019,2009,False,28056.455518,23267.281004,8116.780839,109.0,NaN,NaN,NaN,...,NaN,NaN,NaN,371.288430,325.990546,31.914143,13.383741,NaN,NaN,20149.026584
